# Two-Sample Testing Methods in Scala

This notebook demonstrates the two-sample testing methods available in the RobustInfer Scala library.

In [1]:
%AddJar file:/app/scala_lib/build/libs/robustInfer-scala-0.1.0.jar


Starting download from file:/app/scala_lib/build/libs/robustInfer-scala-0.1.0.jar
Finished download of robustInfer-scala-0.1.0.jar
Using cached version of robustInfer-scala-0.1.0.jar


In [2]:
import robustinfer._
import org.apache.commons.math3.distribution.CauchyDistribution
import scala.util.Random
import org.apache.spark.rdd.RDD


## Basic Two-Sample Tests

Let's start with a simple example comparing two groups.


In [8]:
// Generate simple test data
val rng = new Random(42)
import org.apache.spark.rdd.RDD
import org.apache.spark.SparkContext

def samplePositiveCauchy(dist: CauchyDistribution, n: Int): Seq[Double] =
  Seq.fill(n)(math.max(0.0, dist.sample()))

def simulationCauchy(n: Int, diff: Double): (RDD[Double], RDD[Double]) = {
  val dist1 = new CauchyDistribution(0.0, 1.0)
  val dist2 = new CauchyDistribution(diff, 1.0)
    
  val cauchy1 = samplePositiveCauchy(dist1, n)
  val cauchy2 = samplePositiveCauchy(dist2, n)

  val rdd1 = sc.parallelize(cauchy1)
  val rdd2 = sc.parallelize(cauchy2)
  (rdd1, rdd2)
}

val data = simulationCauchy(n=100, diff=0.9)
val group1 = data._1
val group2 = data._2

// Run all three tests
val tTestResult = TwoSample.tTest(group1, group2)
val mwUResult = TwoSample.mwU(group1, group2)  
val zTUResult = TwoSample.zeroTrimmedU(group1, group2)


rng = scala.util.Random@435e7856
data = (ParallelCollectionRDD[93] at parallelize at <console>:69,ParallelCollectionRDD[94] at parallelize at <console>:70)
group1 = ParallelCollectionRDD[93] at parallelize at <console>:69
group2 = ParallelCollectionRDD[94] at parallelize at <console>:70
tTestResult = (1.7112678...


samplePositiveCauchy: (dist: org.apache.commons.math3.distribution.CauchyDistribution, n: Int)Seq[Double]
simulationCauchy: (n: Int, diff: Double)(org.apache.spark.rdd.RDD[Double], org.apache.spark.rdd.RDD[Double])


(1.7112678...

In [9]:
println("Basic Two-Sample Test Results:")
println(f"t-Test:           statistic = ${tTestResult._1}%7.4f, p-value = ${tTestResult._2}%7.4f")
println(f"Mann-Whitney U:   statistic = ${mwUResult._1}%7.4f, p-value = ${mwUResult._2}%7.4f")
println(f"Zero-Trimmed U:   statistic = ${zTUResult._1}%7.4f, p-value = ${zTUResult._2}%7.4f")

Basic Two-Sample Test Results:
t-Test:           statistic =  1.7113, p-value =  0.0870
Mann-Whitney U:   statistic =  3.6529, p-value =  0.0003
Zero-Trimmed U:   statistic =  3.7637, p-value =  0.0002
